# DAPO: Practical Improvements to GRPO

**DAPO (Decoupled Clip and Dynamic Sampling Policy Optimization)** mainly introduces four practical improvements to GRPO to improve training efficiency, exploration, and stability.

## 1. Dynamic Sampling — Avoid Wasted Computation

**Problem:** If all completions for a prompt receive the same reward, their group-relative advantages are zero.

Example (3 completions per prompt):

| Prompt | Rewards | Advantages | Useful? |
|---|---|---|---|
| Q1 | [1, 0, 0] | Nonzero | Yes |
| Q2 | [1, 1, 1] | [0, 0, 0] | No |
| Q3 | [0, 0, 0] | [0, 0, 0] | No |

**Solution:** Discard zero-variance groups and keep sampling until the batch contains enough groups with nonzero reward variance.

This improves the proportion of useful training samples, although it may require additional rollouts and changes the sampled training distribution.

## 2. Clip-Higher — Preserve Exploration

**Problem:** PPO-style clipping stops providing gradients once the importance ratio exceeds the clipping boundary in the advantage-improving direction.

Example:

- Advantage: $A=+1$
- Importance ratio: $\rho=1.25$
- Standard clipping range: $[0.8,1.2]$

The positive update is clipped, so its surrogate gradient becomes zero.

**Solution:** Use asymmetric clipping:

$$
[1-\epsilon_{\text{low}},1+\epsilon_{\text{high}}]
$$

For example, changing the range to \([0.8,1.28]\) allows the token with \(\rho=1.25\) to keep receiving a gradient.

**Goal:** Reduce premature clipping of positive updates and help prevent entropy collapse.

## 3. Token-Level Policy Gradient Loss — Remove Per-Response Length Normalization

**Problem:** Original GRPO averages token losses separately within each completion:

$$
J_{\text{GRPO}}
=\frac1B\sum_i\frac1{T_i}\sum_t\ell_{i,t}
$$

Example:

| Completion | Length | Per-token normalization coefficient |
|---|---:|---:|
| A | 10 | \(1/(2\times10)=0.05\) |
| B | 500 | \(1/(2\times500)=0.001\) |

Tokens in the shorter completion receive a 50× larger normalization coefficient.

**Solution:** Normalize over all generated tokens in the batch:

$$
\boxed{
J_{\text{DAPO}}=
\frac{\sum_i\sum_t\ell_{i,t}}{\sum_i T_i}
}
$$

For these two completions, every token now has the same coefficient:

$$
\frac1{10+500}=\frac1{510}
$$

**Connection to Dr. GRPO:**

- Dr. GRPO uses a fixed denominator $B L_{\max}$.
- DAPO uses the actual total token count $\sum_i T_i$.

For a fixed batch and identical per-token losses, the gradients differ only by a global scaling factor.

## 4. Overlong Reward Shaping — Handle Truncated Completions

**Problem:** A completion may reach the maximum generation length before producing its final answer. Assigning reward 0 can confuse truncation with incorrect reasoning.

Example: Maximum length = 512 tokens.

**Solution A: Overlong Filtering**

Exclude truncated completions from the policy-gradient loss.

**Solution B: Soft Overlong Punishment**

Gradually penalize responses approaching the length limit.

For example:

$$
R_{\text{length}}=
\begin{cases}
0 & T\le448\\
-\frac{T-448}{64} & 448<T\le512
\end{cases}
$$

This creates a smoother signal discouraging excessively long responses.

## Summary

| Improvement | Problem | Solution |
|---|---|---|
| Dynamic Sampling | Zero-advantage groups waste rollout computation | Resample until enough informative groups are collected |
| Clip-Higher | Positive updates get clipped too early | Increase the upper clipping bound |
| Token-Level Loss | Per-response normalization changes token weighting | Normalize by total generated tokens |
| Overlong Reward Shaping | Truncation creates noisy reward signals | Filter truncated responses or apply soft length penalties |

**Key takeaway:** DAPO is primarily a collection of practical improvements to GRPO rather than a fundamentally new policy-gradient algorithm.

Unlike Dr. GRPO, original DAPO retains group standard-deviation normalization for advantages.

**Next question → CISPO:** Instead of clipping away a token's gradient, can we clip its importance weight while still allowing it to learn?